# Data Audit Starter — `loan_applications.csv`

Companion notebook for the lesson **Auditing Messy Real-World Data**. Work top to bottom; the
`# TODO` cells are yours to fill in. Check your numbers against `loan_applications_answer_key.md`.

> **Run this from the `notebooks/` folder** — the load path is relative (`../datasets/…`).

In [ ]:
import pandas as pd

df = pd.read_csv("../datasets/loan_applications.csv")
print("shape:", df.shape)
print(df.dtypes)
df.head(10)

## Task 1 — first contact
**TODO:** one column is a number stored as text (its dtype is `str`, not a number). Which one?
Write it down — you will parse it later.

In [ ]:
# Task 2 — missing-value map
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame(
    {"missing_count": missing, "missing_pct": missing_pct}
).sort_values("missing_pct", ascending=False)
print(missing_report[missing_report["missing_count"] > 0])

In [ ]:
# TODO (Task 2c) — is the missingness random, or does it cluster?
# Hint: cibil_score may be missing far more often for one employment type.
#   df.groupby("employment_type")["cibil_score"].apply(lambda s: s.isna().mean())

In [ ]:
# TODO — the dtype trap: monthly_emi looks numeric but is text.
#   print(df["monthly_emi"].dtype)
#   pd.to_numeric(df["monthly_emi"], errors="coerce")  # which values refuse to parse?

In [ ]:
# Task 3 — value sanity checks (runs as-is because the amount columns are numeric)
print("Age range:", df["age"].min(), "to", df["age"].max())
print("Negative ages:", (df["age"] < 0).sum())
print("Ages above 100:", (df["age"] > 100).sum())
print("Income negatives:", (df["annual_income"] < 0).sum())
print("Income zeros:", (df["annual_income"] == 0).sum())
df["loan_to_income"] = df["loan_amount"] / df["annual_income"]
print("Loan-to-income > 50:", (df["loan_to_income"] > 50).sum())

In [ ]:
# TODO (Task 3c) — duplicates
#   df["application_id"].duplicated().sum()   # repeated IDs
#   df.duplicated().sum()                      # fully repeated rows

## The timestamp trap
`event_timestamp` was stitched together from three systems (IST, UTC, and a US `mm/dd/yyyy`
source). One column, three realities. The cell below shows how many values a naive parse drops.

In [ ]:
parsed = pd.to_datetime(df["event_timestamp"], errors="coerce")
print("rows a naive parse turns into NaT:", parsed.isna().sum())
print(df["event_timestamp"].head(8).tolist())

In [ ]:
# Task 4 — categorical mess
for col in df.select_dtypes(include="object").columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(10))

In [ ]:
# TODO (Task 4b) — write clean_column() to standardise employment_type:
#   .str.strip().str.lower() + a replacement map for the variants, then reapply to the column.

## Close it out
Write the **Data Quality Report** (worksheet Task 5): total rows, the 3 most critical issues
(name the column, the size, the risk to a loan model), and a usable / not-usable verdict. Then
open `loan_applications_answer_key.md` and compare.